# IQA Phase 3 Lineage Evidence

This notebook is a soutenance and handover support. It does not replace the executable runbooks. It gives a compact reading of the validated Phase 3 evidence: Docker Hub deployment, Airflow container orchestration, DVC/MinIO lineage, model checkpoints in MinIO, MLflow Registry as source of truth, and PostgreSQL metadata boundaries.

## 1. Storage And Governance Boundaries

| Boundary | Responsibility |
| --- | --- |
| Git | Code, tests, docs, configs and lightweight manifests |
| DVC/MinIO | Versioned datasets and replay snapshots, especially `s3://iqa-dvc` |
| Model MinIO | ROI and Feature-AE checkpoints under `s3://iqa-models` |
| MLflow Registry | Source of truth for active/candidate model governance |
| PostgreSQL | Facts, statuses, timestamps, versions, URIs and JSONB payloads |

Heavy artifacts are never stored in PostgreSQL or Git. MinIO stores files; MLflow Registry decides model governance.

## 2. Deployment Evidence

The deployment proof starts from Docker Hub images and avoids rebuilding application images on the server:

```bash
export IQA_IMAGE_REGISTRY=adrien1101
export IQA_IMAGE_TAG=sha-<commit>
export IQA_DOCKER_GID="$(stat -c '%g' /var/run/docker.sock)"

docker login
docker compose -f deploy/docker-compose.yml -f deploy/docker-compose.prod.yml pull
docker compose -f deploy/docker-compose.yml -f deploy/docker-compose.prod.yml up -d
```

Smoke test from the server host shell:

```bash
IQA_API_URL=http://localhost:8000 \
IQA_INFERENCE_URL=http://localhost:8100 \
IQA_MINIO_URL=http://localhost:9000 \
IQA_PROMETHEUS_URL=http://localhost:9090 \
IQA_GRAFANA_URL=http://localhost:3000 \
IQA_MLFLOW_URL=http://localhost:5000 \
IQA_AIRFLOW_URL=http://localhost:8080 \
IQA_GATEWAY_URL=http://localhost \
bash deploy/smoke-test.sh
```

## 3. Airflow Runtime Evidence

Airflow Phase 3 is validated as a container orchestrator. The scheduler imports DAGs and the lightweight factory, while business work runs through container tasks.

Key proof points:

- `iqa_dvc_reproducibility` is a separate DVC gate.
- business DAGs use container tasks instead of direct runtime imports.
- `iqa_gpu` serializes GPU-bound work.
- lifecycle is triggered by data events, not CI.

```bash
docker compose -f deploy/docker-compose.yml -f deploy/docker-compose.prod.yml exec airflow-webserver airflow dags list
docker compose -f deploy/docker-compose.yml -f deploy/docker-compose.prod.yml exec airflow-webserver airflow pools list
docker compose -f deploy/docker-compose.yml -f deploy/docker-compose.prod.yml exec airflow-webserver \
  airflow dags trigger iqa_dvc_reproducibility \
  --conf '{"with_network": false, "skip_regeneration": true}'
```

## 4. Replay Lifecycle Evidence

The replay lifecycle runner demonstrates the operational loop: bootstrap model, incoming lots, ROI segmentation, calibrated Feature-AE scoring, oracle feedback, lifecycle decision, and optional training evidence.

Decision-only proof:

```bash
uv run --extra cu128 iqa-run-replay-lifecycle-cycle \
  --scenario-id production_replay_natural \
  --image-root /opt/iqa/iqa-mlops/data/raw/hss-iad \
  --mode decision-only \
  --max-events 60 \
  --wait-for-gpu
```

Lineage summary:

```bash
uv run --extra cu128 iqa-lineage-summary \
  --replay-run-dir .cache/iqa/replay_lifecycle/production_replay_natural/<run_id> \
  --model-version rd_feature_ae_gated_v001_bootstrap
```

## 5. MLflow Registry Evidence

A short `train-on-trigger` run proves the additional MLflow/Registry link without promoting production automatically.

```bash
uv run --extra cu128 iqa-run-replay-lifecycle-cycle \
  --scenario-id production_replay_natural \
  --image-root /opt/iqa/iqa-mlops/data/raw/hss-iad \
  --mode train-on-trigger \
  --max-events 60 \
  --stage test \
  --epochs 1 \
  --max-steps 5 \
  --wait-for-gpu

uv run --extra cu128 iqa-lineage-summary \
  --replay-run-dir .cache/iqa/replay_lifecycle/production_replay_natural/<run_id> \
  --model-version rd_feature_ae_gated_v001_bootstrap \
  --require-mlflow-run
```

Expected reading: `mlflow_run_id` is present, required tags include dataset, manifest, git commit, scenario and preprocessing contract, and MLflow Registry remains the model source of truth.

In [ ]:
import json
import os
from pathlib import Path

run_dir = os.environ.get("RUN_DIR")
if not run_dir:
    print("Set RUN_DIR to a replay lifecycle run directory to load summary.json.")
else:
    summary_path = Path(run_dir) / "summary.json"
    with summary_path.open(encoding="utf-8") as fh:
        summary = json.load(fh)
    display({
        "scenario_id": summary.get("scenario_id"),
        "mode": summary.get("mode"),
        "events_processed": summary.get("events_processed"),
        "lots_processed": summary.get("lots_processed"),
        "trigger_lifecycle": summary.get("trigger_lifecycle"),
        "trigger_reason": summary.get("trigger_reason"),
        "mlflow_run_id": summary.get("mlflow_run_id"),
        "status": summary.get("status"),
    })

## 6. Demo Personas

- **Sophie** sees predictions and the display-only feedback path.
- **Marc** follows lots, lifecycle triggers, calibrated thresholds and lineage summaries for industrial quality decisions.
- **Laurent** checks access boundaries, auditability, Docker Hub provenance, Airflow container isolation, and separation between API, PostgreSQL metadata, MinIO artifacts, DVC lineage and MLflow Registry governance.

Final message: IQA is not just an ML API. It is a traceable operating loop where data, models, deployments and governance stay separated and auditable.